In [1]:
from dotenv import load_dotenv
load_dotenv()
import os
from supabase import create_client, Client
supabase_url = os.environ.get("SUPABASE_URL")
supabase_api_key = os.environ.get("SUPBASE_KEY")

supabase: Client = create_client(supabase_url, supabase_api_key)

In [2]:
from openai import OpenAI
client = OpenAI(api_key = os.getenv("OPENAI_API_KEY"))

In [21]:
def report_research(date, next_date):
    result = supabase.table('curation_selected_items')\
        .select('event_id, output,sources, news_date','topic, created_at')\
    .gte('created_at', date)\
    .lt('created_at', next_date)\
    .eq('topic', 'Report')\
        .order('created_at', desc=False)\
        .execute()
    
    return result.data

In [26]:
report_deep_dive = report_research('2026-02-26', '2026-02-27')

In [27]:
report_deep_dive

[{'event_id': '1540_2026-02-23',
  'output': 'McKinsey (The State of Organizations 2026): Published a comprehensive report (PDF) summarizing survey results from >10,000 executives and nine major organizational shifts, with a substantial focus on AI—noting widespread experimentation with AI agents, low reported bottom-line gains so far, and urgent recommendations to rewire operating models, invest heavily in people and governance, and treat "agentic AI" adoption as an enterprise transformation priority. This report is high-signal for enterprise policy, governance, and regulatory engagement because it synthesizes large-scale leader sentiment and offers prescriptive guidance that could shape corporate approaches to risk, safety, and AI accountability.',
  'sources': [{'url': 'https://www.mckinsey.com/~/media/mckinsey/business%20functions/people%20and%20organizational%20performance/our%20insights/the%20state%20of%20organizations/2026/the-state-of-organizations-2026.pdf',
    'name': 'McKin

In [28]:
def openai_research_workflow(research_input):
    for i in research_input:
        response = client.responses.create(
            model = "gpt-5-nano",
            tools = [{
                "type": "web_search"
            }],
            include = ["web_search_call.action.sources"],
            input = f"""
            You are a senior research analyst for Krux.

            TASK:
              Create a deep, fact-checked brief for ONE AI report/paper/benchmark release.
            
            GOAL:
              Extract the highest-value findings and implementation guidance so a later 100-word summary can capture most practical signal.
            
            OUTPUT FORMAT:
              - 12 to 16 bullet points.
              - Each bullet must include inline citations:
                [Source: <publisher>, <url>]
              - No text before or after bullets.
            
            MANDATORY STRUCTURE:
              1) REPORT SNAPSHOT
              - What was published, by whom, and when.
              - Scope: geography, sectors, sample size, time window, method type.
            
              2) CORE FINDINGS (MOST IMPORTANT)
              - 4 to 6 bullets with the strongest quantified findings.
              - Include exact numbers, not vague language.
            
              3) WHAT THIS ACTUALLY MEANS
              - 2 to 3 bullets translating findings into plain English implications for real teams.
            
              4) IMPLEMENTATION PLAYBOOK
              - 3 to 4 bullets: what organizations should do in the next quarter.
              - Include sequencing (e.g., data readiness -> pilot -> measurement -> rollout).
            
              5) METRICS TO TRACK
              - 1 or 2 bullets on KPIs teams should monitor to apply the report in practice.
            
              6) LIMITATIONS / CAVEATS
              - 1 or 2 bullets on methodology limits, sample bias, missing data, or uncertainty.
            
              7) DECISION TAKEAWAY
              - 1 bullet: “If a team only does one thing based on this report, it should be X.”
            
              STRICT RULES:
              - No hype, no opinion, no generic AI commentary.
              - Every factual claim must be source-backed.
              - If a key detail is missing, write: "Not disclosed in cited sources."
              - Prefer primary sources (original report/paper) over secondary articles.
              - If secondary coverage conflicts with primary report, prioritize primary and note conflict.
            
              QUALITY CHECK:
              - Must contain quantified findings.
              - Must contain actionable implementation steps.
              - Must clearly separate findings from interpretation.
              - Must include limitations.
            
              Report/event to research:
              Report summary: {i['output']}
              Source: {i['sources']}
            """,
        )

        output = response.output_text
        print(output)

        final_dictionary = {
            'event_id': i['event_id'],
            'news_date': i['news_date'],
            'output': output,
            'model_provider': 'openai',
            'topic': i['topic']
        }
        print(final_dictionary)

        save_research(final_dictionary)

        print("✅ Done")

In [29]:
openai_research_workflow(report_deep_dive)

- REPORT SNAPSHOT: The State of Organizations 2026, McKinsey & Company; published February 19, 2026; a 74-page PDF outlining nine organizational shifts with a strong AI focus. [Source: McKinsey, https://www.mckinsey.com/capabilities/people-and-organizational-performance/our-insights/the-state-of-organizations]

- REPORT SNAPSHOT: Scope includes a survey of more than 10,000 senior executives across 15 countries and 16 industries, positioning this as the second edition (2023 being the first). [Source: McKinsey, https://www.mckinsey.com/capabilities/people-and-organizational-performance/our-insights/the-state-of-organizations]

- REPORT SNAPSHOT: Method type is a cross-sectional executive survey; time window not explicitly disclosed in the cited materials. [Source: McKinsey, https://www.mckinsey.com/capabilities/people-and-organizational-performance/our-insights/the-state-of-organizations]

- REPORT SNAPSHOT: The report centers on nine themes and identifies three tectonic forces shaping o

In [8]:
def save_research(research_json):
    supabase.table('research_assistant').insert({
        'event_id': research_json['event_id'],
        'model_provider': research_json['model_provider'],
        'news_date': research_json['news_date'],
        'output': research_json['output'],
        'topic': research_json['topic']
    }).execute()